# TorchVision Models, Freezing, and Fine-Tuning

Deadline: Tue, April 28

# 🎯 Objective

The goal of this assignment is to:

* get familiar with **pretrained image classification models** available in `torchvision.models`,
* learn how to use the modern **weights API** and the preprocessing attached to pretrained weights,
* inspect and modify the **classification head** of a pretrained model,
* practice **freezing** a pretrained backbone and training only a new classifier,
* compare different **transfer-learning strategies**: frozen backbone, partial fine-tuning, and full fine-tuning,
* understand the tradeoff between **accuracy, number of trainable parameters, and training cost**,
* learn basic **experiment tracking** with **Weights & Biases (W&B)**.

This assignment continues the CNN material from the lecture and introduces a practical transfer-learning workflow used in modern computer vision.


---

# 📌 General Requirements

* Use **PyTorch** and **TorchVision**
* Submit a **Jupyter Notebook (.ipynb)**
* The notebook must:
  * run from top to bottom without errors,
  * include markdown explanations,
  * include plots where required,
  * set a random seed,
  * use clear variable names,
  * be reproducible.


---

# ⚠ Restrictions

✅ Allowed:

* `torch.nn.Module`
* `torchvision.models`
* pretrained TorchVision weights via the modern `weights=...` API
* freezing parameters via `param.requires_grad = False`
* replacing the classifier / final linear layer
* optimizers from `torch.optim`
* `DataLoader`, `torchvision.datasets`, `torchvision.transforms`
* **Weights & Biases (W&B)** for experiment tracking
* plotting libraries such as `matplotlib`

❌ Not allowed:

* high-level frameworks such as Lightning, fastai, or Trainer APIs
* external training code copied without explanation
* AutoML / hyperparameter search tools
* using a pretrained model as a black box without showing how you adapted it

You must still write:

* the model-loading code,
* the classifier replacement code,
* the freezing logic,
* the training loop,
* the validation loop,
* the experiment code,
* the discussion of results.


---

# 📚 Recommended Reading

Before starting, review the following materials:

## Official TorchVision / PyTorch documentation

* [TorchVision models and pre-trained weights](https://docs.pytorch.org/vision/stable/models.html)
* [PyTorch transfer learning tutorial](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
* [`torchvision.models.alexnet`](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.alexnet.html)
* [`torchvision.models.resnet18`](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)
* [`torchvision.models.resnet34`](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet34.html)
* [`torchvision.models.resnet50`](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet50.html)
* [`torchvision.models.resnet152`](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet152.html)
* [`torchvision.models.mobilenet_v3_small`](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.mobilenet_v3_small.html)
* [`torch.nn.Module`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

## Weights & Biases

* [W&B Quickstart](https://docs.wandb.ai/models/quickstart)
* [Track Jupyter notebooks with W&B](https://docs.wandb.ai/models/track/jupyter)
* [W&B integration with PyTorch](https://docs.wandb.ai/models/integrations/pytorch)

## Optional background reading

* [CS231n notes on transfer learning](https://cs231n.github.io/transfer-learning/)


---

# Dataset

Use **CIFAR-10**.

Each sample has shape:

$$
x \in \mathbb{R}^{3 \times 32 \times 32}
$$

However, pretrained TorchVision models were trained with their own expected preprocessing pipeline. Therefore, for the selected pretrained weights, you should use the transform provided by the corresponding weights object.

Example:

```python
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT
preprocess = weights.transforms()
model = resnet18(weights=weights)
```

For simplicity, you may use the preprocessing returned by `weights.transforms()` for both training and validation. If you decide to add augmentation, explain clearly what you changed.


---

# Suggested Models

Choose models from `torchvision.models`. For this assignment, the most practical starting choices are:

* `resnet18`
* `mobilenet_v3_small`
* `efficientnet_b0`

You may also choose one of the following optional models:

* `alexnet`
* `resnet34`
* `resnet50`
* `resnet152`

Notes:

* `alexnet` is acceptable either as a pretrained TorchVision model or, if you prefer, as a simplified version implemented from scratch.
* deeper models such as `resnet50` and especially `resnet152` may be significantly slower and more memory-demanding.
* if you work on CPU, a smaller model such as `mobilenet_v3_small` or `resnet18` may be much more convenient.


---

# 📈 Experiment Tracking with Weights & Biases (strongly recommended)

For this assignment, I strongly recommend using **Weights & Biases (W&B)**.

Why?

* it keeps all hyperparameters and metrics in one place,
* it makes it much easier to compare several runs,
* it stores training curves automatically,
* it helps you avoid confusion when you test multiple heads / settings,
* it is very convenient in a notebook.

You should still include plots in the notebook itself, but W&B is an excellent way to track experiments cleanly.

## Minimal notebook-style usage

```python
# !pip install wandb -q

import wandb

wandb.login()

run = wandb.init(
    project="nn-assignment-6",
    config={
        "model": "resnet18",
        "epochs": 5,
        "batch_size": 64,
        "lr": 1e-3,
        "setup": "frozen_backbone"
    }
)
```

During training, log metrics after each epoch:

```python
wandb.log({
    "epoch": epoch,
    "train_loss": train_loss,
    "val_loss": val_loss,
    "train_acc": train_acc,
    "val_acc": val_acc,
})
```

At the end of the notebook or experiment:

```python
run.finish()
```

## Recommendation

## Important note on W&B evidence

**If you use W&B, you must include at least one visible proof of your experiment tracking in the submitted notebook:**

* **either a screenshot of the dashboard,**
* **or a short note with the run name / link,**
* and you must still keep the required summary plots in the notebook.

If you do not use W&B, your notebook is still acceptable, but you will need to organize your experiments very carefully.


---

# Task 1 — Inspect and adapt pretrained TorchVision models (3 pts)

In this task, you will learn how TorchVision models are constructed and how to adapt them to CIFAR-10.

## Requirements


1. Load **two different pretrained image classification models** from `torchvision.models`.
   * At least one of them must be `resnet18`.
   * The second one can be `mobilenet_v3_small`, `efficientnet_b0`, `alexnet`, `resnet34`, `resnet50`, `resnet152`, or another comparable classification model from TorchVision.
2. For each model:
   * print the architecture,
   * identify where the final classifier layer is located,
   * determine the input dimension of the final classifier,
   * replace the final classifier so that the model outputs **10 logits**.
3. For each model, report:
   * total number of parameters,
   * name / location of the replaced classifier layer,
   * output shape for one mini-batch.
4. Run one mini-batch through each adapted model and verify that the output shape is:

$$
(B, 10)
$$


5. Briefly compare the two models:
   * where the classifier head is located,
   * how many parameters they have,
   * which model seems lighter / heavier,
   * what preprocessing is attached to the default pretrained weights.

## Required discussion

Explain:

* why the original classifier must be replaced,
* why pretrained models need their own preprocessing,
* why different architectures expose the final classifier in different places,
* which of the two models seems more convenient for CIFAR-10 on your hardware.

## Hint

For ResNet, the final layer is typically stored in `model.fc`. For AlexNet, MobileNet, and EfficientNet, the final layer is usually inside `model.classifier`.


---

# Task 2 — Train a frozen-feature baseline on CIFAR-10 (3 pts)

Choose **one** of your adapted pretrained models from Task 1 and train it on CIFAR-10 as a **fixed feature extractor**.

In other words:

* keep the pretrained backbone frozen,
* train only the newly created classifier head.

This is the standard and simplest transfer-learning setup.

## Requirements


1. Load **CIFAR-10** and create a **train / validation** split.
2. Use the preprocessing corresponding to the selected weights.
3. Freeze the backbone and keep only the new classifier head trainable.
4. Train the model for at least **3 epochs**.
5. Use:
   * `CrossEntropyLoss`,
   * an optimizer of your choice from `torch.optim`.
6. Track:
   * training loss,
   * validation loss,
   * training accuracy,
   * validation accuracy.
7. Plot:
   * loss vs epoch,
   * accuracy vs epoch.
8. Report:
   * number of trainable parameters,
   * final validation accuracy.
9. **Strongly recommended:** log the experiment to **W&B**.

## Required explanation

Explain:

* what is frozen and what remains trainable,
* why this setup is called **transfer learning**,
* why this setup is cheaper than training the full model.


---

# Task 3 — Compare freezing vs fine-tuning (4 pts)

Use the **same model** as in Task 2 and compare two simple transfer-learning strategies:

### Model A — frozen backbone

* all pretrained backbone parameters frozen,
* only the new classifier head trainable.

### Model B — full fine-tuning

* all model parameters trainable.

Use the same:

* train / validation split,
* batch size,
* preprocessing,
* number of epochs.

You may use a smaller learning rate for the fine-tuned model.

## Requirements


1. Train both settings.
2. For each setting, report:
   * number of trainable parameters,
   * final validation loss,
   * final validation accuracy.
3. Create one final comparison table with:
   * model setting,
   * trainable parameters,
   * final validation accuracy.
4. Plot validation accuracy for both settings.
5. **Strongly recommended:** track both runs in **W&B**.

## Required discussion

Discuss:

* which setup worked better,
* whether fine-tuning improved the result enough to justify the extra training cost,
* which setup you would choose in practice.

## Practical note

If training on the full dataset is too slow on your hardware, you may use a **smaller subset** of CIFAR-10. In that case:

* clearly state the subset size,
* use exactly the same subset for both compared settings,
* keep the comparison fair.


---

# ✅ Deliverables

Your notebook should contain:

* inspection of **two pretrained TorchVision models**,
* classifier replacement for CIFAR-10,
* one trained **frozen-feature baseline**,
* a comparison of **two transfer-learning strategies** (frozen backbone vs full fine-tuning),
* plots and summary tables,
* written discussion of results,
* preferably a short **W&B** record of your experiments.


---

# Starter Code

```python
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets
from torchvision.models import resnet18, ResNet18_Weights

# optional
import wandb

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

```python
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
```

```python
weights = ResNet18_Weights.DEFAULT
preprocess = weights.transforms()

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=preprocess,
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=preprocess,
)

train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
```

```python
model = resnet18(weights=weights)
num_features = # TODO
model.fc = nn.Linear(num_features, 10)
model = model.to(device)

print(model)
print("Total parameters:", count_parameters(model))
print("Trainable parameters:", count_trainable_parameters(model))
```

```python
# Freezing model parameters
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

print("Trainable parameters after freezing:", count_trainable_parameters(model))
```

```python
# minimal W&B example
run = wandb.init(
    project="nn-assignment-6",
    config={"model": "resnet18", "epochs": 5, "lr": 1e-3}
)

# ... training loop ...
# wandb.log({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})

run.finish()
```